In [1]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
)
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# --- Configuration ---
BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/baania')
START_URL = "https://www.baania.com/s/%E0%B8%97%E0%B8%B1%E0%B9%89%E0%B8%87%E0%B8%AB%E0%B8%A1%E0%B8%94/project?province=3781&sort.updated=desc"
OUTPUT_CSV_FILE = BASE_DIR / 'baania_listing_urls.csv'

MAX_PAGES = 5
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    """Configures Chrome for maximum performance by blocking heavy resources."""
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)


def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    """Extracts unique project listing URLs from Baania."""
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('a[href^="/project/"]'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]


def scroll_to_bottom(driver: uc.Chrome):
    """Scrolls down smoothly to trigger lazy-loaded elements."""
    last_h = 0
    for _ in range(6):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.4)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def click_next_page(driver: uc.Chrome, wait: WebDriverWait) -> bool:
    """Finds and clicks the 'Next Page' button on Baania."""
    try:
        xpath = "//a[@aria-label='Go to next page']"
        next_btn = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))

        # Check if the button is disabled (sometimes indicated by a class on the parent element)
        parent_li = next_btn.find_element(By.XPATH, "..")
        if "disabled" in parent_li.get_attribute("class"):
            return False

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_btn)
        return True
    except (TimeoutException, NoSuchElementException, ElementClickInterceptedException):
        return False


def get_first_listing_href(driver: uc.Chrome) -> str:
    """Helper function to monitor URL changes between page loads."""
    js = "return document.querySelector('a[href^=\"/project/\"]')?.getAttribute('href') || '';"
    return driver.execute_script(js)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    all_urls = set()
    current_page = 1

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)

        while current_page <= MAX_PAGES:
            logging.info(f"Processing Page {current_page}/{MAX_PAGES}...")

            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/project/"]')))
            except TimeoutException:
                logging.warning(f"Timeout waiting for elements on page {current_page}. Stopping.")
                break

            scroll_to_bottom(driver)

            new_links = extract_links(driver, base_url)
            if not new_links:
                logging.info("No links found on this page. Stopping.")
                break

            all_urls.update(new_links)
            logging.info(f"Found {len(new_links)} listings (Total unique: {len(all_urls)})")

            if current_page >= MAX_PAGES:
                logging.info(f"Reached target max pages ({MAX_PAGES}).")
                break

            current_first_href = get_first_listing_href(driver)
            current_url = driver.current_url

            if not click_next_page(driver, wait):
                logging.info("No 'Next' button found or it is disabled. Reached the last page.")
                break

            try:
                wait.until(
                    lambda d: d.current_url != current_url or get_first_listing_href(d) != current_first_href
                )
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/project/"]')))
            except TimeoutException:
                logging.error("Timeout waiting for the next page to load. Stopping.")
                break

            current_page += 1

    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in sorted(all_urls):
            writer.writerow([u])

    logging.info(f"Successfully saved {len(all_urls)} URLs to {OUTPUT_CSV_FILE}")


if __name__ == "__main__":
    main()

2026-03-29 16:24:07,586 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:24:08,895 - INFO - Starting scrape at: https://www.baania.com/s/%E0%B8%97%E0%B8%B1%E0%B9%89%E0%B8%87%E0%B8%AB%E0%B8%A1%E0%B8%94/project?province=3781&sort.updated=desc
2026-03-29 16:24:11,149 - INFO - Processing Page 1/5...
2026-03-29 16:24:12,321 - INFO - Found 48 listings (Total unique: 48)
2026-03-29 16:24:14,388 - INFO - Processing Page 2/5...
2026-03-29 16:24:15,221 - INFO - Found 48 listings (Total unique: 96)
2026-03-29 16:24:16,475 - INFO - Processing Page 3/5...
2026-03-29 16:24:17,377 - INFO - Found 48 listings (Total unique: 144)
2026-03-29 16:24:18,591 - INFO - Processing Page 4/5...
2026-03-29 16:24:19,422 - INFO - Found 48 listings (Total unique: 192)
2026-03-29 16:24:20,672 - INFO - Processing Page 5/5...
2026-03-29 16:24:21,519 - INFO - Found 48 listings (Total unique: 240)
2026-03-29 16:24:21,519 - INFO - Reached target max

In [2]:
import csv
import logging
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/baania')
INPUT_CSV = BASE_DIR / 'baania_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'baania_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-extensions")
    
    return uc.Chrome(options=options, version_main=145)


def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    try:
        title = driver.find_element(By.CSS_SELECTOR, 'h1.titleH1').text.strip()
        data.extend(["--- Listing Title ---", title])
    except NoSuchElementException:
        pass

    try:
        address = driver.find_element(By.CSS_SELECTOR, 'span.address').text.strip()
        data.extend(["\n--- Address ---", address])
    except NoSuchElementException:
        pass

    try:
        details_xpath = "//h2[contains(., 'รายละเอียดโครงการ')]/ancestor::div[contains(@class, 'Section__WrapperTop')]/following-sibling::div[contains(@class, 'collapse')]//div[contains(@class, 'pre-line')]"
        details = driver.find_element(By.XPATH, details_xpath).text.strip()
        if details:
            data.extend(["\n--- Project Details ---", details])
    except NoSuchElementException:
        pass

    try:
        highlights_xpath = "//h4[contains(., 'จุดเด่นของโครงการ')]/following-sibling::div[contains(@class, 'pre-line')]"
        highlights = driver.find_element(By.XPATH, highlights_xpath).text.strip()
        if highlights:
            data.extend(["\n--- Highlights ---", highlights])
    except NoSuchElementException:
        pass

    return "\n".join(data)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 8)

    try:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    if driver.current_url != url:
                        driver.get(url)
                    
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.titleH1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

2026-03-29 16:28:04,634 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:28:16,895 - INFO - Progress: [10/240] scraped.
2026-03-29 16:28:24,683 - INFO - Progress: [20/240] scraped.
2026-03-29 16:28:32,934 - INFO - Progress: [30/240] scraped.
2026-03-29 16:28:40,233 - INFO - Progress: [40/240] scraped.
2026-03-29 16:28:47,150 - INFO - Progress: [50/240] scraped.
2026-03-29 16:28:54,369 - INFO - Progress: [60/240] scraped.
2026-03-29 16:29:02,887 - INFO - Progress: [70/240] scraped.
2026-03-29 16:29:10,660 - INFO - Progress: [80/240] scraped.
2026-03-29 16:29:16,986 - INFO - Progress: [90/240] scraped.
2026-03-29 16:29:29,550 - ERROR - [94/240] Timeout loading: https://www.baania.com/project/ลาซัวว์-เดอ-เอส-68050c03e82363001a952ec9
2026-03-29 16:29:36,659 - INFO - Progress: [100/240] scraped.
2026-03-29 16:29:44,329 - INFO - Progress: [110/240] scraped.
2026-03-29 16:29:53,334 - INFO - Progress: [120/240] scraped.